# 02 · Feature engineering, model development & experiment tracking
This notebook drives the same code as `python -m heartrisk.train`, so what you see here is exactly what CI runs.

Pipeline: `ClinicalFeatureAdder → ColumnTransformer(impute+scale | impute | impute+one-hot) → classifier`.  
Candidates: **Logistic Regression**, **Random Forest**, **Gradient Boosting** — each tuned with grid search (stratified 5-fold, ROC-AUC) *inside* the pipeline so preprocessing is re-fitted per fold.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))   # works even if `pip install -e .` was skipped

import json
import pandas as pd
from IPython.display import Image, display
from heartrisk import config
from heartrisk.data import load_processed
from heartrisk.features import build_pipeline, ClinicalFeatureAdder
from heartrisk.modeling import candidate_models
from heartrisk.tracking import get_tracker
from heartrisk.train import run_training

## 1 · What the engineered features look like

In [ ]:
df = load_processed()
X = df[config.FEATURE_COLUMNS]
ClinicalFeatureAdder().transform(X)[["age", "thalach", "hr_pct_max", "trestbps", "high_bp", "chol", "high_chol", "oldpeak", "st_depressed"]].head()

## 2 · Hyper-parameter search space

In [ ]:
for name, (est, grid) in candidate_models(config.RANDOM_STATE).items():
    n = 1
    for v in grid.values(): n *= len(v)
    print(f"{name:20s} {n:3d} combinations  ->  {grid}")

## 3 · Train, tune, evaluate and log to MLflow
Set `QUICK = True` for a 20-second smoke run; the shipped model uses the full grids (`QUICK = False`).

In [ ]:
QUICK = False
tracker = get_tracker(True, "heart-disease-risk")      # logs to ./mlruns
meta = run_training(quick=QUICK, tracker=tracker)
print("Selected:", meta["model_name"], "| version", meta["model_version"])

## 4 · Model comparison

In [ ]:
cols = ["cv_accuracy_mean", "cv_precision_mean", "cv_recall_mean", "cv_f1_mean", "cv_roc_auc_mean",
        "cv_roc_auc_std", "test_accuracy", "test_precision", "test_recall", "test_roc_auc"]
pd.DataFrame(meta["all_models"]).T[cols].round(3)

In [ ]:
fig_dir = config.FIGURES_DIR / "model"
display(Image(str(fig_dir / "cv_model_comparison.png")))
display(Image(str(fig_dir / "roc_overlay_test.png")))

## 5 · Model-selection logic
1. **Tune** each model with grid search on the training split (5-fold stratified CV, ROC-AUC).
2. **Re-score** the winning configuration on a *different* CV shuffle — `best_score_` is the maximum over many configs and therefore optimistic.
3. **Select** the final model by cross-validated ROC-AUC. The hold-out test set is **never** used for selection, only for the final, unbiased report.
4. Prefer the simpler model when scores are within one standard deviation (interpretability matters in clinical settings).

In [ ]:
display(Image(str(fig_dir / f"importance_{meta['model_name']}.png")))
display(Image(str(fig_dir / f"confusion_{meta['model_name']}.png")))

Inspect all runs in the MLflow UI: `make mlflow-ui` → http://localhost:5000 (select **heart-disease-risk**; each candidate is a child of `training-session`).